**Notebook purpose:**

Objective
Extract zone information from plant codes and add a new `zone` column to both consumption and daily delivery datasets.

Logic
Map plant codes to zones:
- `P_A1`, `P_A2` → `P_A`
- `P_B1` → `P_B`
- `P_C1` → `P_C`

This enables zone-level analysis and reporting across multiple plants.

In [1]:
import pandas as pd
import yaml

In [2]:
with open("../config/config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

---

#### Consumption

In [3]:
consumption_db = pd.read_csv(config["consumption_db_pbi_path"], encoding="utf-8")

In [4]:
consumption_db.info()

<class 'pandas.DataFrame'>
RangeIndex: 19843 entries, 0 to 19842
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   load_id           19843 non-null  int64  
 1   date              19843 non-null  str    
 2   plant             19843 non-null  str    
 3   product           19843 non-null  str    
 4   product_category  19843 non-null  str    
 5   note              4258 non-null   str    
 6   volume            19843 non-null  float64
 7   cement_42_5r      19843 non-null  float64
 8   cement_3a_42      19843 non-null  float64
 9   cement_4av        19843 non-null  float64
 10  arid_04           19843 non-null  float64
 11  arid_8_14         19843 non-null  float64
 12  arid_12_20        19843 non-null  float64
 13  water             19843 non-null  float64
 14  ad_complast_mr    19843 non-null  float64
 15  ad_dynamon_sx     19843 non-null  float64
 16  ad_aut_dynamon    19843 non-null  float64
 17  ad_a

In [5]:
# Define plant-to-zone mapping
plant_to_zone_map = {
    "P_A1": "P_A",
    "P_A2": "P_A",
    "P_B1": "P_B",
    "P_C1": "P_C",
}

# Apply mapping to create zone column
consumption_db["plant_zone"] = consumption_db["plant"].map(plant_to_zone_map)

# Reorder columns: zone comes after date
cols = consumption_db.columns.tolist()
cols.remove("plant_zone")
date_idx = cols.index("date")
cols.insert(date_idx + 1, "plant_zone")
consumption_db = consumption_db[cols]

consumption_db.head()

,load_id,date,plant_zone,plant,product,product_category,note,volume,cement_42_5r,cement_3a_42,cement_4av,arid_04,arid_8_14,arid_12_20,water,ad_complast_mr,ad_dynamon_sx,ad_aut_dynamon,ad_auramix,corrective_02
0,0,2024-01-02,P_A,P_A1,HA-25-B-20,structural_concrete,NaN,3.0,0.0,0.0,831.0,3709.0,1158.0,1566.0,452.0,5.4,4.05,0.0,0.0,0
1,1,2024-01-02,P_A,P_A2,CT-C12-F3,non_structural_concrete,NaN,6.0,0.0,0.0,1510.0,10200.0,0.0,0.0,1289.0,0.0,0.00,16.5,0.0,0
2,2,2024-01-02,P_A,P_A2,CT-C12-F3,non_structural_concrete,NaN,6.0,0.0,0.0,1510.0,10270.0,0.0,0.0,1289.0,0.0,0.00,16.5,0.0,0
3,3,2024-01-02,P_A,P_A2,MD-200-P-4,dosed_mortar,M-5 CON,6.0,0.0,0.0,1202.0,11450.0,0.0,0.0,833.0,7.2,0.00,0.0,0.0,0
4,4,2024-01-02,P_A,P_A2,CT-C12-F3,non_structural_concrete,NaN,6.0,0.0,0.0,1508.0,10190.0,0.0,0.0,1289.0,0.0,0.00,16.5,0.0,0


---

#### Daily_deliveries. 

In [6]:
daily_deliveries_db = pd.read_csv(config["daily_deliveries_db_pbi_path"], encoding="utf-8")

In [7]:
# daily_deliveries.info without () -> different output view.

In [8]:
daily_deliveries_db.info()

<class 'pandas.DataFrame'>
RangeIndex: 9012 entries, 0 to 9011
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   record_id         9012 non-null   int64  
 1   date              9012 non-null   str    
 2   customer          9012 non-null   str    
 3   worksite          9012 non-null   str    
 4   plant             9012 non-null   str    
 5   product           9012 non-null   str    
 6   product_category  9012 non-null   str    
 7   volume            9012 non-null   float64
 8   billing_company   9012 non-null   str    
dtypes: float64(1), int64(1), str(7)
memory usage: 633.8 KB


In [9]:
# Define plant-to-zone mapping
plant_to_zone_map = {
    "P_A": "P_A",
    "P_A3": "P_A",
    "P_B1": "P_B",
    "P_B2": "P_B",
    "P_C1": "P_C",
}

# Apply mapping to create zone column
daily_deliveries_db["plant_zone"] = daily_deliveries_db["plant"].map(plant_to_zone_map)

# Reorder columns: zone comes after date
cols = daily_deliveries_db.columns.tolist()
cols.remove("plant_zone")
date_idx = cols.index("date")
cols.insert(date_idx + 4, "plant_zone")
daily_deliveries_db = daily_deliveries_db[cols]

daily_deliveries_db.head()

,record_id,date,customer,worksite,plant,plant_zone,product,product_category,volume,billing_company
0,0,2024-01-02,COSMA_VITRUVIO_PALLADIO,CASTRA_VEII_FIDENAE_CORFINIUM,P_A,P_A,HA-25-B-20-XC1,structural_concrete,10.0,H
1,1,2024-01-02,COSMA_Y_MODENA_HORTENSIA_ULISSE,PLACENTIA_BRUNDISIUM,P_A,P_A,HM-20-B-20,structural_concrete,7.0,H
2,2,2024-01-02,ETRURIA,CIRCUS_VELEIA,P_A,P_A,CT-C12-F3,non_structural_concrete,33.5,H
3,3,2024-01-02,FLORIANO_Y_ONORATO_HORTENSIA_CAPRI,RUSCINO_MEDULLIA,P_A,P_A,HA-25-B-20,structural_concrete,3.0,H
4,4,2024-01-02,NORCIA_XENOFONTE,DYRRACHIUM_DYRRACHIUM_SPOLETIUM_VEII_ZADAR,P_A3,P_A,MD-200,dosed_mortar,6.0,H


In [10]:
consumption_db.to_csv(config["consumption_db_pbi_csv_path"], index=False, encoding="utf-8")

In [11]:
daily_deliveries_db.to_csv(config["daily_deliveries_db_pbi_csv_path"], index=False, encoding="utf-8")